# Create commands to run Koruami no Below lab server

In [2]:
import glob
import os
import pandas as pd

In [ ]:
%%bash
# Save the docker run into a bash file for easier setup in slurm
# Since I direct the output to a log file, need to create the folder first
imgt_version=3.64
sampleid=R299996754
bam_path=/data/bams/workspace_extraction_40001-45000/R299996754.unsorted.extract.bam
# kourami_db=/data/kourami_reference/imgt_hla_db/${imgt_version}
# output_path=/data/output/kourami_run_${imgt_version}/${sampleid}
docker run --rm \
--mount type=bind,src=/DC3/AGD250k_HLA_typing/extracted_hla_read_bams,dst=/data/bams,readonly \
--mount type=bind,src=/data100t1/home/wanying/shared_data_files/Illumina_reference,dst=/data/reference,readonly \
--mount type=bind,src=/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/data/kourami_reference,dst=/data/kourami_reference,readonly \
--mount type=bind,src=/DC3/AGD250k_HLA_typing/individual_hla_results,dst=/data/output \
--mount type=bind,src=/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code,dst=/data/code \
kourami-hla-tools:1.0 \
bash -c "mkdir -p ${output_path} && bash /data/code/utils/run_kourami.skip_hla_read_extraction.sh ${imgt_version} ${sampleid} ${bam_path}> ${output_path}/${sampleid}.log"

# 1. One example run
I put the code in a bash file `/data/code/utils/run_kourami.skip_hla_read_extraction.sh`.
Need to run it using the Kourami docker image.

# 2. Create commands to run in slurm

In [ ]:
%%bash
imgt_version=3.64
sampleid=R299996754
bam_path=/data/bams/workspace_extraction_40001-45000/R299996754.unsorted.extract.bam
bash /data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code/utils/docker_run_kourami.skip_read_extraction.sh ${imgt_version} ${sampleid} ${bam_path}

## 2.1 Docker commands

In [31]:
# Create folder names
bam_path = '/DC3/AGD250k_HLA_typing/extracted_hla_read_bams'
lst_bam_folder = [f'{bam_path}/workspace_extraction_{i}-{i+5000-1}' for i in range(1, 250002, 5000)] + [f'{bam_path}/rerun_failed_ones']
dict_fh = dict()
c = 0
for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
    fn_output = f'./bash_slurm/docker_kourami_runs.agd250k.imgt_v{imgt_version}.sh'
    dict_fh[imgt_version] = open(fn_output, 'w')

for path in lst_bam_folder:
    for bam_path in glob.glob(f'{path}/*bam'):
        bam_path = bam_path.replace('/DC3/AGD250k_HLA_typing/extracted_hla_read_bams', '/data/bams')
        c += 1
        sample_id = bam_path.split('/')[-1].split('.')[0]
        for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
            cmd = f'imgt_version={imgt_version}; sampleid={sample_id}; bam_path={bam_path}; '
            cmd += 'bash /data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code/utils/docker_run_kourami.skip_read_extraction.sh ${imgt_version} ${sampleid} ${bam_path}'
            dict_fh[imgt_version].write(cmd+'\n')

        if c%1000 ==0:
            print(f'\r# Total cmd written: {c}', end='', flush=True)
print(f'\r# Total cmd written: {c}')

for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
    dict_fh[imgt_version].close()
print('# Done')


# Total cmd written: 250283
# Done


## 2.2 Singularity commands
I do not have permission to use docker deamon on some servers, so slurm jobs failed on those servers.
Singularity seems to be universal for all severs.

In [17]:
# Check if singularity run works. Some server does not have issues with docker.
bam_path = '/DC3/AGD250k_HLA_typing/extracted_hla_read_bams'
lst_bam_folder = [f'{bam_path}/workspace_extraction_{i}-{i+5000-1}' for i in range(1, 250002, 5000)] + [f'{bam_path}/rerun_failed_ones']
dict_fh = dict()
c = 0
for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
    fn_output = f'./bash_slurm/singularity_kourami_runs.agd250k.imgt_v{imgt_version}.sh'
    dict_fh[imgt_version] = open(fn_output, 'w')

for path in lst_bam_folder:
    for bam_path in glob.glob(f'{path}/*bam'):
        bam_path = bam_path.replace('/DC3/AGD250k_HLA_typing/extracted_hla_read_bams', '/data/bams')
        c += 1
        sample_id = bam_path.split('/')[-1].split('.')[0]
        for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
            cmd = f'imgt_version={imgt_version}; sampleid={sample_id}; bam_path={bam_path}; '
            cmd += 'bash /data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code/utils/singularity_run_kourami.skip_read_extraction.sh ${imgt_version} ${sampleid} ${bam_path}'
            dict_fh[imgt_version].write(cmd+'\n')

        if c%1000 ==0:
            print(f'\r# Total cmd written: {c}', end='', flush=True)
print(f'\r# Total cmd written: {c}')

for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
    dict_fh[imgt_version].close()
print('# Done')


# Total cmd written: 250283
# Done


## 2.3 Bash run in parallel with `xargs`
Somehow the slurm jobs got cut (maybe server issue?). So I plan to run docker commands in screen sessions in parallel

In [ ]:
%%bash
# Run this in a screen session
cd /data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code/bash_slurm
command_file=docker_kourami_runs.agd250k.imgt_v3.24.sh
n_jobs_parallel=16 # Run this number of jobs in parallel, the number of CPU we have for vgipiper06 is 64, running 32 jobs already max out the server
cat ${command_file} | xargs -d '\n' -P ${n_jobs_parallel} -I {} bash -c "{}"

## 2.4 Run multiple samples in docker -it mode
Docker and sigularity runs are killed at somepoint without error message. Not sure what was wrong.

So try start a container and run in -it mode

```
docker run \
--mount type=bind,src=/DC3/AGD250k_HLA_typing/extracted_hla_read_bams,dst=/data/bams,readonly \
--mount type=bind,src=/data100t1/home/wanying/shared_data_files/Illumina_reference,dst=/data/reference,readonly \
--mount type=bind,src=/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/data/kourami_reference,dst=/data/kourami_reference,readonly \
--mount type=bind,src=/DC3/AGD250k_HLA_typing/individual_hla_results,dst=/data/output \
--mount type=bind,src=/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code,dst=/data/code \
-it wanyingzhu/kourami-hla-tools:1.0 /bin/bash
```

### 2.4.1 With additional loci `-a` flag

In [2]:
# Create cmds to run in docker container
# eg: bash docker_it_run_kourami.sh 3.64 R299996754 /data/bams/workspace_extraction_40001-45000/R299996754.unsorted.extract.bam

bam_path = '/DC3/AGD250k_HLA_typing/extracted_hla_read_bams'
lst_bam_folder = [f'{bam_path}/workspace_extraction_{i}-{i+5000-1}' for i in range(1, 250002, 5000)] + [f'{bam_path}/rerun_failed_ones']
dict_fh = dict()
c = 0
for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
    fn_output = f'./bash_slurm/docker_it_kourami_runs.agd250k.imgt_v{imgt_version}.sh'
    dict_fh[imgt_version] = open(fn_output, 'w')

for path in lst_bam_folder:
    for bam_path in glob.glob(f'{path}/*bam'):
        bam_path = bam_path.replace('/DC3/AGD250k_HLA_typing/extracted_hla_read_bams', '/data/bams')
        c += 1
        sample_id = bam_path.split('/')[-1].split('.')[0]
        for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
            cmd = f'imgt_version={imgt_version}; sampleid={sample_id}; bam_path={bam_path}; '
            cmd += 'bash /data/code/utils/docker_it_run_kourami.sh ${imgt_version} ${sampleid} ${bam_path}'
            dict_fh[imgt_version].write(cmd+'\n')

        if c%1000 ==0:
            print(f'\r# Total cmd written: {c}', end='', flush=True)
print(f'\r# Total cmd written: {c}')

for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
    dict_fh[imgt_version].close()
print('# Done')



# Total cmd written: 250283
# Done


In [8]:
# Split the commands in to subset to speed up the run
def chunk_cmd(cmd_fn, imgt_version, chunk_size=5000):
    c, file_indx = 0, 1
    with open(cmd_fn) as fh:
        output_fn = f'./bash_slurm/docker_it_v{imgt_version}/cmd_subset_{(file_indx-1)*chunk_size+1}-{file_indx*chunk_size}.sh'
        output_fh = open(output_fn, 'w')
        for line in fh:
            c += 1
            if c==chunk_size:
                output_fh.write(line)
                output_fh.close()
                # reset
                c = 0
                file_indx += 1
                output_fn = f'./bash_slurm/docker_it_v{imgt_version}/cmd_subset_{(file_indx-1)*chunk_size+1}-{file_indx*chunk_size}.sh'
                output_fh = open(output_fn, 'w')
            else:
                output_fh.write(line)
    print(f'# Files created: {file_indx}')
    output_fh.close()

# chunk_cmd(cmd_fn=f'./bash_slurm/docker_it_kourami_runs.agd250k.imgt_v3.64.sh',
#           imgt_version='3.64', chunk_size=5000)

for imgt_version in ['3.24', '3.59', '3.64']: 
    chunk_cmd(cmd_fn=f'./bash_slurm/docker_it_kourami_runs.agd250k.imgt_v{imgt_version}.sh',
              imgt_version=imgt_version, chunk_size=5000)

# Files created: 51
# Files created: 51
# Files created: 51


In [ ]:
%%bash
# Run this in a screen session with the docker container running
cd /data/code/bash_slurm/docker_it_v3.64
command_file=cmd_subset_1-5000.sh
n_jobs_parallel=16 # Run this number of jobs in parallel, the number of CPU we have for vgipiper06 is 64, running 32 jobs already max out the server
cat ${command_file} | xargs -d '\n' -P ${n_jobs_parallel} -I {} bash -c "{}"

## 2.5 Run without additional loci `-a` flag
Many runs were stuck, likely due to the additional loci request, especially for IMGT/HLA v3.64

In [ ]:
# Singularity runs
bam_path = '/DC3/AGD250k_HLA_typing/extracted_hla_read_bams'
lst_bam_folder = [f'{bam_path}/workspace_extraction_{i}-{i+5000-1}' for i in range(1, 250002, 5000)] + [f'{bam_path}/rerun_failed_ones']
dict_fh = dict()
c = 0
for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
    fn_output = f'./bash_slurm/singularity_kourami_runs.agd250k.imgt_v{imgt_version}.no_additional_loci.sh'
    dict_fh[imgt_version] = open(fn_output, 'w')

for path in lst_bam_folder:
    for bam_path in glob.glob(f'{path}/*bam'):
        bam_path = bam_path.replace('/DC3/AGD250k_HLA_typing/extracted_hla_read_bams', '/data/bams')
        c += 1
        sample_id = bam_path.split('/')[-1].split('.')[0]
        for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
            cmd = f'imgt_version={imgt_version}; sampleid={sample_id}; bam_path={bam_path}; '
            cmd += 'bash /data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code/utils/singularity_run_kourami.skip_read_extraction.no_additional_loci.sh ${imgt_version} ${sampleid} ${bam_path}'
            dict_fh[imgt_version].write(cmd+'\n')

        if c%1000 ==0:
            print(f'\r# Total cmd written: {c}', end='', flush=True)
print(f'\r# Total cmd written: {c}')

for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
    dict_fh[imgt_version].close()
print('# Done')


# Total cmd written: 250283
# Done


In [3]:
# Create folder names
bam_path = '/DC3/AGD250k_HLA_typing/extracted_hla_read_bams'
lst_bam_folder = [f'{bam_path}/workspace_extraction_{i}-{i+5000-1}' for i in range(1, 250002, 5000)] + [f'{bam_path}/rerun_failed_ones']
dict_fh = dict()
c = 0
for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
    fn_output = f'./bash_slurm/docker_kourami_runs.agd250k.imgt_v{imgt_version}.no_additional_loci.sh'
    dict_fh[imgt_version] = open(fn_output, 'w')

for path in lst_bam_folder:
    for bam_path in glob.glob(f'{path}/*bam'):
        bam_path = bam_path.replace('/DC3/AGD250k_HLA_typing/extracted_hla_read_bams', '/data/bams')
        c += 1
        sample_id = bam_path.split('/')[-1].split('.')[0]
        for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
            cmd = f'imgt_version={imgt_version}; sampleid={sample_id}; bam_path={bam_path}; '
            cmd += 'bash /data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code/utils/docker_run_kourami.skip_read_extraction.no_additional_loci.sh ${imgt_version} ${sampleid} ${bam_path}'
            dict_fh[imgt_version].write(cmd+'\n')

        if c%1000 ==0:
            print(f'\r# Total cmd written: {c}', end='', flush=True)
print(f'\r# Total cmd written: {c}')

for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
    dict_fh[imgt_version].close()
print('# Done')


# Total cmd written: 250283
# Done


In [4]:
# Docker -it mode
# Create cmds to run in docker container
# eg: bash docker_it_run_kourami.sh 3.64 R299996754 /data/bams/workspace_extraction_40001-45000/R299996754.unsorted.extract.bam

bam_path = '/DC3/AGD250k_HLA_typing/extracted_hla_read_bams'
lst_bam_folder = [f'{bam_path}/workspace_extraction_{i}-{i+5000-1}' for i in range(1, 250002, 5000)] + [f'{bam_path}/rerun_failed_ones']
dict_fh = dict()
c = 0
for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
    fn_output = f'./bash_slurm/docker_it_kourami_runs.agd250k.imgt_v{imgt_version}.no_additional_loci.sh'
    dict_fh[imgt_version] = open(fn_output, 'w')

for path in lst_bam_folder:
    for bam_path in glob.glob(f'{path}/*bam'):
        bam_path = bam_path.replace('/DC3/AGD250k_HLA_typing/extracted_hla_read_bams', '/data/bams')
        c += 1
        sample_id = bam_path.split('/')[-1].split('.')[0]
        for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
            cmd = f'imgt_version={imgt_version}; sampleid={sample_id}; bam_path={bam_path}; '
            cmd += 'bash /data/code/utils/docker_it_run_kourami.no_additional_loci.sh ${imgt_version} ${sampleid} ${bam_path}'
            dict_fh[imgt_version].write(cmd+'\n')

        if c%1000 ==0:
            print(f'\r# Total cmd written: {c}', end='', flush=True)
print(f'\r# Total cmd written: {c}')

for imgt_version in ['3.24', '3.59', '3.64']: # Create 3 files to use different IMGT/HLA database versions
    dict_fh[imgt_version].close()
print('# Done')



# Total cmd written: 250283
# Done


# 3. Check which samples are completed
The last row in the sample_id.log is "# DONE"

In [3]:
def find_processed_samples(result_path = '/DC3/AGD250k_HLA_typing/individual_hla_results/kourami_run_3.24'):
    '''
    Search the output folder and check done_kourami.log file for successed runs (The file itself is empty)
    '''
    lst_done_samples, lst_need_rerun_samples = [], []
    c_total, c_failed = 0, 0
    for sample_id in os.listdir(result_path):
        c_total += 1
        log_fn = f'{result_path}/{sample_id}/done_kourami.log'
        if not os.path.isfile(log_fn):
            lst_need_rerun_samples.append(sample_id)
            c_failed += 1
        else:
            lst_done_samples.append(sample_id)
            
        if c_total%50==0:
            print(f'\r# N samples failed/checked: {c_failed}/{c_total}', end='', flush=True)
    print(f'\r# N samples failed/checked: {c_failed}/{c_total}')
    return lst_need_rerun_samples, lst_done_samples

def create_commands_to_run(imgt_version:str='3.24',
                           skip_samples:list=[],
                           wrapper_code:str='/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code/utils/singularity_run_kourami.skip_read_extraction.sh',
                           output_prefix:str='rerun_singularity_kourami_runs.agd250k',
                           timeout:str='10m'):
    '''
    Create commands to rerun failed or missed ones, or any desired samples. Auto kill the run with timeout
    Params:
    - imgt_versions: a string of imgt/hla database version to create the commands for
    - skip_samples: a list of samples to skip the run. If empty, will generate commands based on all bam files available
    - wrapper_code: the code with the singularity or docker run, either singularity_run_kourami.skip_read_extraction.sh or singularity_run_kourami.skip_read_extraction.no_additional_loci.sh
    - timeout: 10m=kill the run after 10 minutes (used by the timeout bash command)
    Result
    Save result for each gene in separate files
    '''
    fn_bam_path = '/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/data/agd250k_hla_bam_paths.txt'
    df_bam_path = pd.read_csv(fn_bam_path, sep='\t')

    dict_fh = dict()
    print('# Save cmd to output file(s):')
    fn_output = f'./bash_slurm/{output_prefix}.imgt_v{imgt_version}.sh'
    print(f'# - {fn_output}')
    
    if skip_samples != []:
        df_bam_path = df_bam_path[~df_bam_path['grid'].isin(skip_samples)].copy()
    # Change file path so they match the container mounted dir
    df_bam_path['bam_path_container'] = df_bam_path['bam_path'].apply(lambda x: x.replace('/DC3/AGD250k_HLA_typing/extracted_hla_read_bams', '/data/bams'))
    # Notice the reruns have different file path: /DC3/AGD250k_HLA_typing/extracted_hla_read_bams/rerun_failed_ones/R203773148.extract.bam

    df_bam_path['CMD'] = f"timeout {timeout} bash -c '" + f'imgt_version={imgt_version}; sampleid='+ df_bam_path['grid'] + '; bam_path='+ df_bam_path['bam_path_container'] + '; '
    df_bam_path['CMD'] += f'bash {wrapper_code}' + ' ${imgt_version} ${sampleid} ${bam_path}' + "'"
    df_bam_path[['CMD']].to_csv(fn_output, header=False, index=False)
    print('# N CMD written:', len(df_bam_path))
    print('# Done')


In [20]:
# Write commands to run kourami
for imgt_version in ['3.59', '3.64']:
    create_commands_to_run(imgt_version=imgt_version,
                           skip_samples=[],
                           wrapper_code='/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code/utils/singularity_run_kourami.skip_read_extraction.no_additional_loci.sh',
                           output_prefix='singularity_kourami_runs.agd250k.no_additional_loci',
                           timeout='10m')

# Save cmd to output file(s):
# - ./bash_slurm/singularity_kourami_runs.agd250k.no_additional_loci.imgt_v3.59.sh
# N CMD written: 250283
# Done
# Save cmd to output file(s):
# - ./bash_slurm/singularity_kourami_runs.agd250k.no_additional_loci.imgt_v3.64.sh
# N CMD written: 250283
# Done


## 3.1 Check IMGT/HLA database v3.24

In [ ]:
lst_need_rerun_samples, lst_done_samples = find_processed_samples(result_path = '/DC3/AGD250k_HLA_typing/individual_hla_results/kourami_run_3.24')
print(len(lst_need_rerun_samples), len(lst_done_samples))


create_commands_to_run(imgt_version='3.24',
                       skip_samples=lst_need_rerun_samples+lst_done_samples,
                       wrapper_code='/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code/utils/singularity_run_kourami.skip_read_extraction.sh',
                       output_prefix='rerun_singularity_kourami_runs.agd250k')

create_commands_to_run(imgt_version='3.24',
                       skip_samples=lst_done_samples,
                       wrapper_code='/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code/utils/singularity_run_kourami.skip_read_extraction.no_additional_loci.sh',
                       output_prefix='rerun_failed_singularity_kourami_runs.no_additional_loci.agd250k')

# Save cmd to output file(s):
# - ./bash_slurm/rerun_singularity_kourami_runs.agd250k.imgt_v3.24.sh
# N CMD written: 0
# Done
# Save cmd to output file(s):
# - ./bash_slurm/rerun_failed_singularity_kourami_runs.no_additional_loci.agd250k.imgt_v3.24.sh
# N CMD written: 103
# Done


## 3.2 Check IMGT/HLA database v3.59

In [12]:
lst_need_rerun_samples, lst_done_samples = find_processed_samples(result_path = '/DC3/AGD250k_HLA_typing/individual_hla_results/kourami_run_3.59')
print(len(lst_need_rerun_samples), len(lst_done_samples))

# N samples failed/checked: 1175/250166
1175 248991


In [ ]:
# create_commands_to_run(imgt_version='3.59',
#                        skip_samples=lst_need_rerun_samples+lst_done_samples,
#                        wrapper_code='/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code/utils/singularity_run_kourami.skip_read_extraction.sh',
#                        output_prefix='rerun_singularity_kourami_runs.agd250k')
                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
create_commands_to_run(imgt_version='3.59',
                       skip_samples=lst_done_samples,
                       wrapper_code='/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code/utils/singularity_run_kourami.skip_read_extraction.no_additional_loci.sh',
                       output_prefix='rerun_failed_singularity_kourami_runs.no_additional_loci.agd250k',
                       timeout='10m')

# Save cmd to output file(s):
# - ./bash_slurm/rerun_failed_singularity_kourami_runs.no_additional_loci.agd250k.imgt_v3.59.sh
# N CMD written: 1650
# Done


## 3.3 Check IMGT/HLA database v3.64

In [5]:
lst_need_rerun_samples, lst_done_samples = find_processed_samples(result_path = '/DC3/AGD250k_HLA_typing/individual_hla_results/kourami_run_3.64')
print(len(lst_need_rerun_samples), len(lst_done_samples))

# N samples failed/checked: 1243/250166
1243 248923


In [7]:
create_commands_to_run(imgt_version='3.64',
                       skip_samples=lst_done_samples,
                       wrapper_code='/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/code/utils/singularity_run_kourami.skip_read_extraction.no_additional_loci.sh',
                       output_prefix='rerun_failed_singularity_kourami_runs.no_additional_loci.agd250k',
                       timeout='10m')

# So only need to rerun failed ones

# Save cmd to output file(s):
# - ./bash_slurm/rerun_failed_singularity_kourami_runs.no_additional_loci.agd250k.imgt_v3.64.sh
# N CMD written: 1243
# Done


216.0